# 30 — Validation without training
Lightweight checks for the audited input, alignment, feature chronology, and model interfaces. Synthetic tensors in this notebook are **CODE VALIDATION ONLY** and cannot be reported as research results. No optimizer step runs.

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.data.features import build_features
from src.data.windows import create_rolling_windows
from src.data.scaling import TimeSeriesScaler
from src.data.splits import chronological_split
from src.regimes.rules import detect_regimes
from src.portfolio.volatility_targeting import apply_volatility_target
from src.backtest.backtester import Backtester

## Synthetic alignment checks — CODE VALIDATION ONLY
A window ending at t must pair with t+1. Changing a future price must leave all earlier feature rows unchanged. The scaler must use training rows only.

In [ ]:
dates = pd.bdate_range('2021-01-01', periods=180)
prices = pd.DataFrame({'A': 100 * np.exp(np.arange(180) * .001), 'B': 80 * np.exp(np.arange(180) * .0005)}, index=dates)
features, returns, vol, _ = build_features(prices)
changed = prices.copy(); changed.iloc[-1, 0] *= 1.25
changed_features, _, _, _ = build_features(changed)
pd.testing.assert_frame_equal(features.iloc[:-1], changed_features.iloc[:-1])
train, val, test = chronological_split(features)
assert train.index.max() < val.index.min() < test.index.min()
scaler = TimeSeriesScaler(); scaler.fit(train.to_numpy())
assert np.allclose(scaler.mean, train.to_numpy().mean(axis=0))
X, y, decisions = create_rolling_windows(scaler.transform(features.to_numpy()), returns.to_numpy(), 20, features.index)
assert X.shape[1:] == (20, features.shape[1]) and y.shape[1] == prices.shape[1]
assert (features.index[20:] > decisions).all()
assert np.allclose(y, returns.loc[features.index[20:]].to_numpy())
print('Synthetic chronology, scaler, feature causality, and target alignment checks passed.')

## Position and return checks — CODE VALIDATION ONLY
The backtester receives decision-time signals, decision-time volatility, and next-period returns. Transaction costs must reduce net return by turnover times basis points.

In [ ]:
signals = np.full((6, 2), .5); ex_ante = np.full((6, 2), .2); next_ret = np.array([[.01, -.01]] * 6)
bt = Backtester(target_vol=.1, max_leverage=3, tc_bps=5).run(signals, ex_ante, next_ret)
assert len(bt['net_returns']) == 6
assert np.allclose(bt['gross_returns'] - bt['net_returns'], bt['turnover'] * 5 / 10000)
assert np.nanmax(np.abs(bt['positions'].to_numpy())) <= 3
print('Synthetic backtest arithmetic checks passed.')

## Optional audited-data checks
This cell reads the source CSV when present and checks duplicates and selected column names. It does not repair the data or train a model.

In [ ]:
default_data = ROOT / 'data' / 'kaggle_global_indian_markets' / 'global-india-markets-macro' / 'daily_market_data.csv'
data_path = Path(os.getenv('FIN_DATA_PATH', str(default_data))).resolve()
if data_path.is_file():
    raw = pd.read_csv(data_path)
    date_col = next((c for c in raw if c.lower() == 'date'), None)
    assert date_col is not None, 'Date column must be identified in the audit'
    parsed = pd.to_datetime(raw[date_col], errors='coerce')
    assert parsed.notna().all() and parsed.is_monotonic_increasing and parsed.is_unique
    selected = [s.strip() for s in os.getenv('FIN_ASSETS', '').split(',') if s.strip()]
    canonical = {'NIFTY50','SP500','USDINR','GOLD','BRENT'}
    assert set(selected).issubset(canonical)
    print('Audited raw chronology passed; selected assets:', selected)
else:
    print('Audited CSV unavailable; synthetic code checks only.')

## Review before reporting
Inspect missingness and price units; verify macro release timing; confirm cost convention, leverage-cap frequency, regime threshold provenance, untouched test selection, Mamba implementation identity, multi-seed coverage, and full input hashes. These require human review beyond the synthetic checks.

## Required test and archive gates
Run the repository tests and verify every completed runner unit has table schemas, predictions, matching checkpoints, data provenance, and archive inputs. Any failed check stops notebook execution.

In [ ]:
import json, re, subprocess
import torch
subprocess.run([sys.executable,'-m','pytest','-q','tests'],cwd=ROOT,check=True)
run_id = os.getenv('RUN_ID','').strip()
if not run_id: raise ValueError('RUN_ID is required for archive validation')
output_root = Path(os.getenv('OUTPUT_ROOT',str(ROOT/'outputs'))).resolve()
assert output_root.is_relative_to(ROOT)
run_dir = output_root/run_id
required_inputs = [
    ROOT/'data'/'kaggle_global_indian_markets'/'global-india-markets-macro'/'daily_market_data.csv',
    ROOT/'data'/'kaggle_global_indian_markets'/'global-india-markets-macro'/'monthly_macro_data.csv',
    ROOT/'data'/'kaggle_global_indian_markets'/'global-india-markets-macro'/'codes_and_sources.md',
    run_dir/'metadata'/'run_manifest.json',run_dir/'metadata'/'data_manifest.json',
    run_dir/'metadata'/'dependency_lock.txt',
    run_dir/'metadata'/'config_snapshot.json',
    *(ROOT/'notebooks'/f'{name}.ipynb' for name in ('00_core','10_data_audit','20_benchmark','30_validation','40_report'))]
missing = [str(path) for path in required_inputs if not path.is_file()]
if missing: raise FileNotFoundError('Archive-required inputs missing: '+', '.join(missing))
manifest = json.loads((run_dir/'metadata'/'data_manifest.json').read_text(encoding='utf-8'))
assert manifest['schema_version']=='1.0' and set(manifest['files'])=={'daily','monthly'}
for record in manifest['files'].values():
    assert record['sha256'] and record['columns'] and record['dtypes'] and record['missing_by_column']
markers = sorted((run_dir/'telemetry').glob('*.done'))
if not markers: raise AssertionError('No completed model/seed/fold units')
for marker in markers:
    match = re.fullmatch(r'(.+)__seed_(\d+)__fold_(\d+)\.done',marker.name)
    if not match: raise AssertionError(f'Unexpected completion marker: {marker.name}')
    model, seed, fold = match.group(1),int(match.group(2)),int(match.group(3))
    table = run_dir/'tables'/model/f'seed_{seed}'/f'fold_{fold}'
    required = {'metrics.csv':{'model','seed','fold','cost_bps','Sharpe Ratio','mse','directional_accuracy'},
                'regimes.csv':{'model','seed','fold','regime','sharpe','sharpe_ci_lower','sharpe_ci_upper'},
                'returns.csv':{'model','seed','fold','decision_date','target_date','gross_return'},
                'predictions.csv':{'model','seed','fold','asset','signal','target_return'}}
    for filename, fields in required.items():
        path=table/filename
        if not path.is_file(): raise FileNotFoundError(path)
        frame=pd.read_csv(path)
        if not fields.issubset(frame.columns) or frame.empty: raise AssertionError(f'Invalid {path}')
        if not (frame['model'].astype(str)==model).all() or not (frame['seed']==seed).all() or not (frame['fold']==fold).all():
            raise AssertionError(f'Unit identity mismatch: {path}')
    checkpoint_dir=run_dir/'checkpoints'/model/f'seed_{seed}'
    for filename in ('latest.pt','best.pt'):
        candidate=checkpoint_dir/filename
        if candidate.is_file():
            payload=torch.load(candidate,map_location='cpu',weights_only=False)
            if payload.get('unit_identity',{}).get('fold') != fold:
                candidate=checkpoint_dir/f'fold_{fold}'/filename
        if not candidate.is_file(): raise FileNotFoundError(candidate)
        payload=torch.load(candidate,map_location='cpu',weights_only=False)
        if payload.get('unit_identity') != {'model':model,'seed':seed,'fold':fold}:
            raise AssertionError(f'Checkpoint identity mismatch: {candidate}')
print('Pytest, data manifest, unit schemas, checkpoints and archive inputs passed.')